# Importando as Bilbiotecas

In [16]:
# Manipulação de dados
import re
import numpy as np
import pandas as pd

# Visualização
import matplotlib.pyplot as plt

# NLP - NLTK
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer

# Recursos do NLTK
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("rslp", quiet=True)

# Preparação e validação
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    learning_curve
)

# Representação textual
from sklearn.feature_extraction.text import CountVectorizer

# Modelos
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# Pipeline
from sklearn.pipeline import Pipeline

# Avaliação
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay
)

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


## Leitura da Base de Dados

In [2]:
dados = pd.read_csv("../data/dados_spam.csv", index_col=0)
dados.head()

,categoria,texto
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


# 1. Pré-Processamento dos Dados

### 1.1 Limpeza e Normalização do Texto

In [3]:
ABBREVIATIONS = {
    # Conversa / SMS
    "pls": "please", "plz": "please", "thx": "thanks", "ty": "thank you", "u": "you", "ur": "your", "r": "are",
    "ya": "you", "msg": "message", "txt": "text", "idk": "i do not know", "imo": "in my opinion", "imho": "in my humble opinion",
    "asap": "as soon as possible", "rn": "right now", "btw": "by the way", "fyi": "for your information", "brb": "be right back",
    "ttyl": "talk to you later", "lol": "laughing out loud", "omg": "oh my god",
    # Tempo / urgência
    "tmr": "tomorrow", "tmrw": "tomorrow", "tdy": "today", "2day": "today", "2nite": "tonight", "nite": "night", "wk": "week",
    "wks": "weeks", "hr": "hour", "hrs": "hours", "min": "minute", "mins": "minutes",
    # Financeiro
    "acct": "account", "accts": "accounts", "txn": "transaction", "txns": "transactions", "transac": "transaction", "amt": "amount",
    "bal": "balance", "avl": "available", "stmt": "statement", "stmts": "statements", "ref": "reference",
    # Pagamentos
    "pymt": "payment", "pymts": "payments", "pmt": "payment", "pmts": "payments", "paymt": "payment",
    # Promoções / spam
    "promo": "promotion", "promos": "promotions", "disc": "discount", "pct": "percent", "exp": "expires", "expd": "expired",
    "offer": "offer", "offers": "offers",
    # Contato
    "ph": "phone", "tel": "telephone", "mob": "mobile", "addr": "address", "info": "information",
}

In [4]:
def limpar_texto(texto):

    # 1. Lowercase
    texto = texto.lower()
    # 2. Remover HTML
    texto = re.sub(r'<[^>]+>', " ", texto)
    # 3. Remover URLs
    texto = re.sub(r'https?://\S+|www\.\S+', " ", texto)
    # 4. Remover E-mails
    texto = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', " ", texto)
    # 5. Remover telefones
    texto = re.sub(r'\+?\d[\d\s\-\(\)]{7,}\d', " ", texto)
    # 6. Remover Dinheiro
    texto = re.sub(r'(r\$|\$|€|£)\s?\d+(?:[.,]\d+)?', " ", texto)
    # 7. Substituindo as abreviações
    palavras = [ABBREVIATIONS.get(p,p) for p in texto.split()]
    texto = " ".join(palavras)
    # 8. Remove caracteres indesejados
    texto = re.sub(r'[^a-záàâãéêèíìîóôõöúùûüç\s.,!?]', " ", texto)
    # 9. Normaliza espaços
    texto = re.sub(r'\s+', " ", texto)

    return texto

In [5]:
df_teste = dados.copy()

df_teste["texto_limpo"] = df_teste["texto"].apply(limpar_texto)

df_teste[["texto", "texto_limpo"]].head()

,texto,texto_limpo
0,"Go until jurong point, crazy.. Available only ...","go until jurong point, crazy.. available only ..."
1,Ok lar... Joking wif u oni...,ok lar... joking wif you oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in a wkly comp to win fa cup final ...
3,U dun say so early hor... U c already then say...,you dun say so early hor... you c already then...
4,"Nah I don't think he goes to usf, he lives aro...","nah i don t think he goes to usf, he lives aro..."


### 1.2 Tokenização

In [6]:
def tokenizar(texto):

    return word_tokenize(texto, language="english")

In [7]:
df_teste["texto_token"] = df_teste["texto_limpo"].apply(tokenizar)

df_teste[["texto_limpo", "texto_token"]].head()

,texto_limpo,texto_token
0,"go until jurong point, crazy.. available only ...","[go, until, jurong, point, ,, crazy, .., avail..."
1,ok lar... joking wif you oni...,"[ok, lar, ..., joking, wif, you, oni, ...]"
2,free entry in a wkly comp to win fa cup final ...,"[free, entry, in, a, wkly, comp, to, win, fa, ..."
3,you dun say so early hor... you c already then...,"[you, dun, say, so, early, hor, ..., you, c, a..."
4,"nah i don t think he goes to usf, he lives aro...","[nah, i, don, t, think, he, goes, to, usf, ,, ..."


In [9]:
# Pegando 1 exemplo
print(df_teste["texto_token"][0])

['go', 'until', 'jurong', 'point', ',', 'crazy', '..', 'available', 'only', 'in', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', '...', 'cine', 'there', 'got', 'amore', 'wat', '...']


### 1.3 Removendo StopWords

In [10]:
STOP_WORDS = set(stopwords.words("english"))

NEGACAO = {
    "no", "not", "nor", "never", "neither",
    "none", "nobody", "nothing", "nowhere",
    "cannot", "without",

    # Contrações negativas
    "don't", "doesn't", "didn't",
    "isn't", "aren't", "wasn't", "weren't",
    "haven't", "hasn't", "hadn't",
    "won't", "wouldn't", "can't", "couldn't",
    "shouldn't", "mustn't", "needn't",
}

STOP_WORDS_EN = STOP_WORDS - NEGACAO

In [13]:
def stop_words(tokens):
    resultado = []
    for token in tokens:
        if (token.isalpha() and token not in STOP_WORDS_EN):
            resultado.append(token)
    
    return resultado

In [14]:
df_teste["texto_stop_word"] = df_teste["texto_token"].apply(stop_words)

df_teste[["texto_token", "texto_stop_word"]].head()

,texto_token,texto_stop_word
0,"[go, until, jurong, point, ,, crazy, .., avail...","[go, jurong, point, crazy, available, bugis, n..."
1,"[ok, lar, ..., joking, wif, you, oni, ...]","[ok, lar, joking, wif, oni]"
2,"[free, entry, in, a, wkly, comp, to, win, fa, ...","[free, entry, wkly, comp, win, fa, cup, final,..."
3,"[you, dun, say, so, early, hor, ..., you, c, a...","[dun, say, early, hor, c, already, say]"
4,"[nah, i, don, t, think, he, goes, to, usf, ,, ...","[nah, think, goes, usf, lives, around, though]"


In [15]:
# mostrando exemplos
print(df_teste["texto_token"][0])
print(df_teste["texto_stop_word"][0])

['go', 'until', 'jurong', 'point', ',', 'crazy', '..', 'available', 'only', 'in', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', '...', 'cine', 'there', 'got', 'amore', 'wat', '...']
['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'wat']


### 1.4 Stemmer

In [17]:
stemmer = SnowballStemmer("english")

def aplicar_stemmer(tokens):
    resultado =[]
    for token in tokens:
        resultado.append(stemmer.stem(token))

    return resultado

In [18]:
df_teste["texto_stemmer"] = df_teste["texto_stop_word"].apply(aplicar_stemmer)

df_teste[["texto_stop_word", "texto_stemmer"]].head()

,texto_stop_word,texto_stemmer
0,"[go, jurong, point, crazy, available, bugis, n...","[go, jurong, point, crazi, avail, bugi, n, gre..."
1,"[ok, lar, joking, wif, oni]","[ok, lar, joke, wif, oni]"
2,"[free, entry, wkly, comp, win, fa, cup, final,...","[free, entri, wkli, comp, win, fa, cup, final,..."
3,"[dun, say, early, hor, c, already, say]","[dun, say, earli, hor, c, alreadi, say]"
4,"[nah, think, goes, usf, lives, around, though]","[nah, think, goe, usf, live, around, though]"


In [19]:
# mostrando exemplos
print(df_teste["texto_stop_word"][0])
print(df_teste["texto_stemmer"][0])

['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'wat']
['go', 'jurong', 'point', 'crazi', 'avail', 'bugi', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amor', 'wat']


### 1.5 Aplicando Pipeline Completo

In [20]:
def pipeline_completo(texto):

    texto = limpar_texto(texto)
    token = tokenizar(texto)
    stop_word = stop_words(token)
    stemmer = aplicar_stemmer(stop_word)

    return " ".join(stemmer)

In [21]:
dados["texto_tratado"] = dados["texto"].apply(pipeline_completo)
dados.head()

,categoria,texto,texto_tratado
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,ok lar joke wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entri wkli comp win fa cup final tkts st ...
3,ham,U dun say so early hor... U c already then say...,dun say earli hor c alreadi say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


# 2. Preparação dos Dados

### 2.1 Divisão dos Dados

In [22]:
X = dados["texto_tratado"]
y = dados["categoria"]

In [23]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

print(f"Total Treino -> {len(X_treino)}")
print(f"Total Teste -> {len(X_teste)}")

Total Treino -> 4459
Total Teste -> 1115


### 2.2 Vetorização com BoW + Ngram

In [24]:
vectorizer = CountVectorizer(
    lowercase=True,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.85,
    max_features=10000
)

# 3. Modelos NLP